In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

def start_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--lang=en-US")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
   
    options.page_load_strategy = 'eager' 
    
    driver = webdriver.Chrome(options=options)
    
   
    driver.set_page_load_timeout(25)
    
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver

def set_amazon_ny_location(driver):
    print("📍 جاري محاولة ضبط الموقع تلقائياً على الولايات المتحدة (نيويورك 10001)...")
    try:
        wait = WebDriverWait(driver, 15)
        location_btn = wait.until(EC.element_to_be_clickable((By.ID, "nav-global-location-slot")))
        location_btn.click()
        time.sleep(3)
        
        try:
            zip_input = wait.until(EC.presence_of_element_located((By.ID, "GLUXZipUpdateInput")))
            zip_input.clear()
            zip_input.send_keys("10001")
            time.sleep(1)
            
            apply_btn = driver.find_element(By.ID, "GLUXZipUpdate")
            apply_btn.click()
            time.sleep(3)
            
            try:
                done_btn = driver.find_element(By.NAME, "glowDoneButton")
                done_btn.click()
            except:
                driver.refresh()
                
            time.sleep(5)
            print("🇺🇸 تم تثبيت موقع نيويورك (10001) بنجاح واختفاء رسالة الشحن لمصر!")
            return True
        except Exception as e:
            print("⚠️ واجهة الرمز البريدي لم تفتح مباشرة، جاري الاعتماد على الكوكيز الافتراضية.")
            return False
    except Exception as e:
        print("❌ فشل ضبط الموقع تلقائياً:", e)
        return False


base_url = "https://www.amazon.com/s?i=electronics&rh=n:172659&s=popularity-rank&page="
data = []


driver = start_driver()

try:
    
    driver.get("https://www.amazon.com")
    time.sleep(4)
    driver.add_cookie({"name": "lc-main", "value": "en_US", "domain": ".amazon.com"})
    set_amazon_ny_location(driver)
    
    
    for page in range(1, 31):
        print(f"\n🔵 جاري سحب بيانات الصفحة رقم {page}...")
        
        try:
            driver.get(base_url + str(page))
        except Exception as timeout_error:
           
            print(f"⚠️ تحميل الصفحة {page} استغرق وقتاً طويلاً، جاري التجميد وسحب المتاح حالياً...")
            driver.execute_script("window.stop();")
            
        time.sleep(random.uniform(4, 6))
        
        
        if "Something went wrong" in driver.page_source or "api-services-support@amazon.com" in driver.page_source:
            print(f"⚠️ ظهرت صفحة حماية أو خطأ في الصفحة {page} → جاري التخطي")
            continue
            
        soup = BeautifulSoup(driver.page_source, "html.parser")
        products = soup.find_all("div", {"data-component-type": "s-search-result"})
        print(f"📦 تم العثور على {len(products)} منتج في هذه الصفحة.")
        
        
        if len(products) == 0:
            print("⚠️ الصفحة فارغة، جاري عمل تحديث سريع...")
            try: 
                driver.refresh()
            except: 
                driver.execute_script("window.stop();")
            soup = BeautifulSoup(driver.page_source, "html.parser")
            products = soup.find_all("div", {"data-component-type": "s-search-result"})
            if len(products) == 0:
                print("❌ لا تزال فارغة → الانتقال للصفحة التالية")
                continue
                
       
        for p in products:
            try:
                
                title = None
                title_tag = p.select_one("h2 a span")
                if title_tag:
                    title = title_tag.get_text(strip=True)
                if not title and p.h2:  
                    title = p.h2.get_text(strip=True)
                if not title:
                    title_tag_alt = p.find("img", class_="s-image")
                    title = title_tag_alt.get("alt", strip=True) if title_tag_alt else None

                
                link = None
                link_tag = p.select_one("h2 a")
                if not link_tag:
                    link_tag = p.find("a", class_="a-link-normal")
                
                if link_tag and link_tag.has_attr('href'):
                    href_val = link_tag["href"]
                   
                    link = href_val if href_val.startswith("http") else "https://www.amazon.com" + href_val

               
                price_tag = p.select_one("span.a-price span.a-offscreen")
                price = price_tag.get_text(strip=True) if price_tag else None
                
               
                old_price_tag = p.select_one("span.a-text-price span.a-offscreen")
                old_price = old_price_tag.get_text(strip=True) if old_price_tag else None
                
                
                rating_tag = p.find("span", class_="a-icon-alt")
                rating = rating_tag.get_text(strip=True) if rating_tag else None
                
                
                sales_tag = p.find("span", class_="a-color-secondary")
                sales_text = sales_tag.get_text(strip=True) if sales_tag else None
                monthly_sales = sales_text if (sales_text and "bought" in sales_text) else "N/A"
                
                
                options_tag = p.select_one("span.s-variation-options-text")
                options = options_tag.get_text(strip=True).replace("Options:", "").strip() if options_tag else "1 Option"
                
                
                if title:
                    data.append({
                        "Title": title,
                        "Price_USD": price if price else "Check Website",
                        "Old_Price_USD": old_price if old_price else "No Discount",
                        "Rating": rating if rating else "No Ratings",
                        "Monthly_Sales": monthly_sales,
                        "Available_Options": options,
                        "Link": link if link else "N/A"
                    })
            except Exception as item_error:
                
                continue

        
        time.sleep(random.uniform(2, 4))

finally:
    
    if data:
        df = pd.DataFrame(data)
        
        df.dropna(subset=["Title"], inplace=True)
        
        output_file = "amazon_US_Premium_Data.xlsx"
        df.to_excel(output_file, index=False)
        print(f"\n✅ تم الحفظ بنجاح! اسم الملف الحالي: {output_file}")
        print(f"📊 إجمالي المنتجات المستخرجة بداتا كاملة ولينكات شغالة بنسبة 100%: {len(df)} منتج.")
    else:
        print("\n❌ لم يتم تجميع بيانات، يرجى مراجعة اتصال الإنترنت أو إعدادات الحماية.")
        
    driver.quit()

📍 جاري محاولة ضبط الموقع تلقائياً على الولايات المتحدة (نيويورك 10001)...
🇺🇸 تم تثبيت موقع نيويورك (10001) بنجاح واختفاء رسالة الشحن لمصر!

🔵 جاري سحب بيانات الصفحة رقم 1...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 2...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 3...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 4...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 5...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 6...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 7...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 8...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 9...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 10...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 11...
📦 تم العثور على 27 منتج في هذه الصفحة.

🔵 جاري سحب بيانات الصفحة رقم 12...
📦 تم العث

In [1]:
import pandas as pd
import sqlite3

# قراءة ملف Excel
df = pd.read_excel("amazon_US_Premium_Data.xlsx")

# حفظ CSV
df.to_csv(
    "amazon_US_Premium_Data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV created")

# إنشاء قاعدة البيانات
conn = sqlite3.connect("amazon_US_Premium_Data.db")

# إنشاء الجدول وإدخال البيانات
df.to_sql(
    "amazon_products",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Database created")

CSV created
Database created


In [2]:
import pandas as pd

df = pd.read_excel("amazon_US_Premium_Data.xlsx")

# حفظ JSON
df.to_json(
    "amazon_US_Premium_Data.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("JSON created")

JSON created


In [3]:
pip install pandas openpyxl sqlalchemy pymysql

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.9 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.1 MB 2.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 2.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.9 MB/s  0:00:01

   ---------------------------------------- 0/3 [pymysql]
   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sq


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from sqlalchemy import create_engine, text

# ===== بيانات الاتصال =====
MYSQL_USER = "root"
MYSQL_PASSWORD = "1234567"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"

DATABASE_NAME = "amazon_db"
TABLE_NAME = "amazon_products"

# ===== قراءة ملف Excel =====
df = pd.read_excel("amazon_US_Premium_Data.xlsx")

# ===== الاتصال بـ MySQL بدون قاعدة بيانات =====
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}"
)

# ===== إنشاء قاعدة البيانات إذا لم تكن موجودة =====
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}"))
    conn.commit()

print("Database created successfully")

# ===== الاتصال بقاعدة البيانات =====
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{DATABASE_NAME}"
)

# ===== إنشاء الجدول واستيراد البيانات =====
df.to_sql(
    TABLE_NAME,
    con=engine,
    if_exists="replace",
    index=False
)

print(f"Table '{TABLE_NAME}' created and data imported successfully")

Database created successfully
Table 'amazon_products' created and data imported successfully


In [6]:
import pandas as pd

# قراءة ملف Excel
df = pd.read_excel("amazon_US_Premium_Data.xlsx")

# اسم ملف SQL الناتج
sql_file = "amazon_products.sql"

# فتح الملف للكتابة
with open(sql_file, "w", encoding="utf-8") as f:

    # إنشاء قاعدة البيانات
    f.write("CREATE DATABASE IF NOT EXISTS amazon_db;\n")
    f.write("USE amazon_db;\n\n")

    # إنشاء الجدول
    f.write("""
CREATE TABLE IF NOT EXISTS amazon_products (
    id INT AUTO_INCREMENT PRIMARY KEY,
    product_name TEXT,
    price TEXT,
    rating TEXT,
    product_url TEXT
);\n\n
""")

    # إدخال البيانات
    for _, row in df.iterrows():

        product_name = str(row[0]).replace("'", "''")
        price = str(row[1]).replace("'", "''")
        rating = str(row[2]).replace("'", "''")
        product_url = str(row[3]).replace("'", "''")

        f.write(
            f"INSERT INTO amazon_products "
            f"(product_name, price, rating, product_url) "
            f"VALUES ('{product_name}', '{price}', '{rating}', '{product_url}');\n"
        )

print("SQL file created successfully ✔")

SQL file created successfully ✔


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6696\1385941369.py:30: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  product_name = str(row[0]).replace("'", "''")
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6696\1385941369.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  price = str(row[1]).replace("'", "''")
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6696\1385941369.py:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  rating = 